# Bobby Carrot RL Training (Colab T4 GPU)

Trains the Bobby Carrot agent for a **single level** using PPO + expert-guided Behavioral Cloning (BC) pre-training.

**How it works:**
- If expert moves exist for the chosen level, BC pre-training runs first to give the agent a warm start on the correct path. PPO then fine-tunes with low entropy so the policy locks in rather than exploring away from it.
- If no expert data is available for the level (levels 9, 12, 21, 25, 26, 29), pure PPO exploration runs with higher entropy.

**Checkpoints are saved to Google Drive** so they survive session resets. Mount Drive in the next cell before running.

| Group | Levels | Expert data | Expected timesteps |
|-------|--------|-------------|--------------------|
| Short path | 1–3, 8 | Yes (21–31 steps) | ~100k |
| Medium path | 4–7, 10–11, 13–16, 20, 24 | Yes (42–80 steps) | ~200k |
| Long path | 17–19, 22–23, 27–28, 30 | Yes (69–187 steps) | ~300–500k |
| No expert data | 9, 12, 21, 25, 26, 29 | No | 500k+ (pure PPO) |

In [ ]:
!pip install pygame torch numpy --quiet

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted. Checkpoints will be saved to /content/drive/MyDrive/bobby_carrot_checkpoints/")

In [ ]:
!git clone https://github.com/Charan20510/RL_Game_Project.git /content/RL_Game_Project

In [ ]:
%cd /content/RL_Game_Project

In [ ]:
from Bobby_Carrot.rl_models.config import LevelConfig, PPOConfig, TrainingConfig
from Bobby_Carrot.rl_models.ppo import train_ppo
from Bobby_Carrot.rl_models.expert_data import EXPERT
from pathlib import Path

################################################
# 1. SET THE SPECIFIC LEVEL YOU WANT TO TRAIN
################################################
LEVEL_KIND = "normal"
LEVEL_NUMBER = 1  # Change this to train different levels separately

# Auto-scale timesteps based on expert path length.
# Longer expert paths → more complex levels → more PPO time needed after BC.
# Levels without expert data use the fallback (500k) for pure PPO exploration.
_expert = EXPERT.get(LEVEL_NUMBER, [])
_path_len = len(_expert)
if _path_len == 0:
    _timesteps = 500_000   # No expert data — pure PPO needs more exploration
elif _path_len <= 35:
    _timesteps = 100_000   # Short path (L1–L3, L8)
elif _path_len <= 80:
    _timesteps = 200_000   # Medium path (L4–L7, L10–L11, L13–L16, L20, L24)
else:
    _timesteps = 400_000   # Long path (L17–L19, L22–L23, L27–L28, L30)

# Training Configuration
# When expert data exists for this level, BC pre-training runs first and entropy
# automatically starts at entropy_min (0.08) to preserve the BC-learned policy.
# For levels without expert data, pure PPO exploration runs with entropy_coeff=0.20
# decaying to entropy_min=0.08.
ppo_config = PPOConfig(
    lr=3e-4,
    clip_ratio=0.2,
    n_epochs=4,
    rollout_length=4096,
    minibatch_size=128,
    entropy_coeff=0.20,  # Used only when no expert data is available for this level
    entropy_min=0.08
)

DRIVE_CKPT = Path("/content/drive/MyDrive/bobby_carrot_checkpoints")
train_config = TrainingConfig(
    total_timesteps=_timesteps,
    device="cuda",
    checkpoint_dir=DRIVE_CKPT / f"{LEVEL_KIND}_{LEVEL_NUMBER}",
    log_dir=DRIVE_CKPT / "logs",
    curriculum=False,
    max_steps_per_episode=1500
)

expert_moves = EXPERT.get(LEVEL_NUMBER, [])
print(f"\n=== Starting PPO Training for Level {LEVEL_KIND} {LEVEL_NUMBER} ===")
if expert_moves:
    print(f"[BC] Expert moves available ({len(expert_moves)} steps) — BC pre-training will run first")
else:
    print("[BC] No expert moves for this level — using pure PPO exploration")
print(f"[PPO] Total timesteps: {_timesteps:,}")

single_level_config = LevelConfig(
    train_levels=[(LEVEL_KIND, LEVEL_NUMBER)],
    test_levels=[(LEVEL_KIND, LEVEL_NUMBER)],
)

train_ppo(ppo_config, train_config, single_level_config, expert_moves=expert_moves)
print(f"=== Finished Training for Level {LEVEL_KIND} {LEVEL_NUMBER} ===")
